# 아이디어 1 파일럿 (2) — Elastic Net: **무엇을 보고 맞혔는가**

앞선 [릿지 노트북](pilot_idea1_ridge.ipynb) 은 "발현으로 CRISPR 의존성을 예측할 수 있는가" 에 답했습니다.
답은 예였습니다. 이 노트북의 질문은 다릅니다.

> **모델이 어떤 유전자를 보고 그렇게 판단했는가?**

릿지는 19,215개 유전자에 **전부 조금씩** 가중치를 줍니다. 성능은 좋지만 "무엇 때문인지" 를 읽을 수 없습니다.
Elastic Net 은 대부분의 가중치를 **정확히 0으로** 만듭니다. 남는 소수가 곧 근거입니다.

| | 릿지 | Elastic Net |
|---|---|---|
| 가중치 | 전부 0이 아님 | **대부분 0** (희소) |
| 속도 | 11.5초 | 263초 (**23배 느림**) |
| 성능 | 중앙값 r **0.404** | 0.354 (**릿지가 더 낫다**) |
| 해석 | 어려움 | **가능** ← 유일한 장점 |

**결론을 먼저 말하면: 성능으로는 릿지가 낫습니다.** 그럼에도 Elastic Net 을 돌리는 이유는
해석 때문입니다. 아래에서 실제 숫자로 확인합니다.

> 이 노트북도 [릿지 노트북](pilot_idea1_ridge.ipynb) 과 같은 이유로 출력을 지우지 않고 커밋합니다
> (ADNI 미사용, 공개 데이터인 DepMap 만 사용).

- 전체 665개 유전자 계산은 [`scripts/pilot_idea1_enet.py`](../scripts/pilot_idea1_enet.py) 가 4.4분에 걸쳐 이미 해두었습니다.
  여기서는 **결과를 읽고**, 시연은 소수 유전자로만 직접 돌립니다.
- 실행 시간: 2분 이내.

## 1. 준비

In [1]:
import sys, time
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from joblib import Parallel, delayed

here = Path.cwd()
repo = next(p for p in [here, *here.parents] if (p / "src").is_dir())
sys.path.insert(0, str(repo))

from src.preprocessing.depmap_io import load_all
from src.models.elastic_net import enet_preselect_cv, top_correlated
from src.models.metrics import pearson_cols
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings("ignore")     # ElasticNet 수렴 경고 (아래 6장에서 다룸)

SD_CUT, N_FOLDS, SEED, TOP_FEAT = 0.25, 5, 0, 200      # 파일럿 스크립트와 동일한 설정
print("저장소:", repo)

저장소: /home/kali/adni-shared/AI-bio-proj-team-1


## 2. 전체 결과 먼저 보기

스크립트가 665개 유전자를 전부 계산해 둔 결과입니다. 릿지와 나란히 놓고 봅니다.

In [2]:
TAB = repo / "results" / "tables"
enet  = pd.read_csv(TAB / "pilot_idea1_enet_gene_scores.csv").set_index("gene")
ridge = pd.read_csv(TAB / "pilot_idea1_ridge_gene_scores.csv").set_index("gene")

cmp = pd.DataFrame({
    "Elastic Net": enet["r_enet"],
    "릿지":         ridge["r_ridge"],
    "암종만":       enet["r_lineage"],
    "자기발현만":   enet["r_self"],
}).dropna(subset=["Elastic Net", "릿지"])

display(pd.DataFrame({
    "중앙값 r":   cmp.median().round(3),
    "r>0.3 비율": (cmp > 0.3).mean().round(3),
    "r>0.5 개수": (cmp > 0.5).sum(),
}))
print(f"두 방법 모두 채점된 유전자: {len(cmp):,}")

,중앙값 r,r>0.3 비율,r>0.5 개수
Elastic Net,0.354,0.689,110
릿지,0.404,0.831,124
암종만,0.226,0.270,24
자기발현만,0.044,0.077,15


두 방법 모두 채점된 유전자: 634


In [3]:
diff = (cmp["Elastic Net"] - cmp["릿지"])
print(f"Elastic Net 이 릿지보다 나은 유전자: {(diff>0).sum():,} / {len(diff):,}  ({(diff>0).mean()*100:.1f}%)")
print(f"두 방법의 상관: {cmp['Elastic Net'].corr(cmp['릿지']):.3f}  (거의 같은 유전자를 맞힌다는 뜻)")

Elastic Net 이 릿지보다 나은 유전자: 92 / 634  (14.5%)
두 방법의 상관: 0.929  (거의 같은 유전자를 맞힌다는 뜻)


### 읽는 법

두 방법의 유전자별 점수는 상관이 0.93 으로 매우 높습니다. **거의 같은 유전자를 맞히고 있다**는 뜻입니다.
다만 전체적으로 릿지가 조금씩 더 잘 맞아서, 중앙값에서 뚜렷한 차이가 납니다.

그러면 23배 느린 Elastic Net 을 왜 돌릴까요? 다음 장이 그 답입니다.

> **주의 — 공정한 모델 비교는 아닙니다.** 릿지는 19,215개 특징을 전부 쓰지만,
> Elastic Net 은 속도 때문에 상관 상위 500개만 남기고 적합합니다.
> 즉 「Elastic Net 이 원리적으로 나쁘다」가 아니라 「이 설정에서는 릿지가 낫다」가 정확한 표현입니다.

## 3. 데이터 준비 — 릿지 노트북과 동일

In [4]:
d = load_all()
targets = d.selective_targets(sd_cut=SD_CUT)

X = d.expression.values.astype(np.float64)
X = (X - X.mean(0)) / (X.std(0) + 1e-8)
gene_names = np.array(d.expression.columns)

lineage = d.lineage
strat = lineage.where(lineage.map(lineage.value_counts()) >= N_FOLDS, "RARE")
splits = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED)
              .split(np.zeros(len(d.gene_effect)), strat))

print(f"세포주 {X.shape[0]:,}  발현 유전자 {X.shape[1]:,}  맞힐 유전자 {len(targets):,}")

세포주 1,140  발현 유전자 19,215  맞힐 유전자 665


## 4. 시연 — 잘 맞은 유전자 6개만 직접 돌려 보기

665개 전부는 12분이 걸리므로, 결과가 좋았던 6개만 골라 계수를 들여다봅니다.

In [5]:
demo = cmp.sort_values("Elastic Net", ascending=False).head(6).index.tolist()
for g in demo:
    print(f"  {g:20s} Elastic Net r={cmp.loc[g,'Elastic Net']:.3f}   암종만 r={cmp.loc[g,'암종만']:.3f}")

  SOX10 (6663)         Elastic Net r=0.837   암종만 r=0.686
  PAX8 (7849)          Elastic Net r=0.795   암종만 r=0.630
  IRF4 (3662)          Elastic Net r=0.780   암종만 r=0.644
  FAM50A (9130)        Elastic Net r=0.778   암종만 r=0.302
  EBF1 (1879)          Elastic Net r=0.774   암종만 r=0.592
  MYB (4602)           Elastic Net r=0.762   암종만 r=0.690


In [6]:
from sklearn.linear_model import ElasticNetCV

# 한 유전자에 대해 적합하고 0이 아닌 계수를 돌려준다.
# 성능 평가가 아니라 해석이 목적이므로 여기서는 전체 데이터를 쓴다.
# (성능 수치는 2장의 교차검증 결과를 봐야 한다.)
def fit_and_explain(gene):
    warnings.filterwarnings("ignore")     # 워커 프로세스에서도 경고 끄기
    y = d.gene_effect[gene].values.astype(np.float64)
    ok = ~np.isnan(y)
    keep = top_correlated(X[ok], y[ok], TOP_FEAT)
    m = ElasticNetCV(l1_ratio=0.5, n_alphas=10, cv=3, max_iter=1000, tol=1e-3,
                     random_state=SEED, n_jobs=1).fit(X[ok][:, keep], y[ok])
    nz = np.flatnonzero(m.coef_)
    return pd.Series(m.coef_[nz], index=gene_names[keep][nz]).sort_values(key=abs, ascending=False)

t0 = time.time()
coefs = dict(zip(demo, Parallel(n_jobs=6)(delayed(fit_and_explain)(g) for g in demo)))
print(f"6개 적합 {time.time()-t0:.0f}초")

6개 적합 2초


## 5. 결과 — 모델이 근거로 삼은 유전자

각 타깃마다 **선택된 특징 수**와 **가중치 상위 5개**를 봅니다.
`self` 표시는 그 유전자 자기 자신의 발현입니다.

In [7]:
for g in demo:
    c = coefs[g]
    print(f"\n■ {g}   — 19,215개 중 {len(c)}개만 사용")
    for name, w in c.head(5).items():
        mark = "  ← 자기 자신" if name == g else ""
        print(f"     {w:+.4f}  {name}{mark}")


■ SOX10 (6663)   — 19,215개 중 45개만 사용
     -0.2129  SOX10 (6663)  ← 자기 자신
     -0.0262  FGFBP2 (83888)
     -0.0260  CDH19 (28513)
     -0.0207  PRSS33 (260429)
     -0.0195  CTXND1 (100996492)

■ PAX8 (7849)   — 19,215개 중 59개만 사용
     -0.0504  PAX8 (7849)  ← 자기 자신
     -0.0364  CLEC4E (26253)
     -0.0284  SEC14L6 (730005)
     -0.0254  KLHL14 (57565)
     -0.0234  RHEX (440712)

■ IRF4 (3662)   — 19,215개 중 47개만 사용
     -0.2082  IRF4 (3662)  ← 자기 자신
     +0.0461  IGLL5 (100423062)
     -0.0429  OR2C1 (4993)
     -0.0376  TNFRSF17 (608)
     -0.0353  JCHAIN (3512)

■ FAM50A (9130)   — 19,215개 중 19개만 사용
     +0.3902  FAM50B (26240)
     -0.0329  RASSF1 (11186)
     -0.0181  DAZAP1 (26528)
     -0.0114  ODC1 (4953)
     +0.0098  WFDC2 (10406)

■ EBF1 (1879)   — 19,215개 중 57개만 사용
     -0.0422  VPREB3 (29802)
     -0.0347  TLR10 (81793)
     -0.0297  MS4A1 (931)
     +0.0229  CD53 (963)
     -0.0185  RAG2 (5897)

■ MYB (4602)   — 19,215개 중 29개만 사용
     -0.0798  RAG2 (5897)
     -0.0722  MY

### 읽는 법 — 이게 파이프라인 검증입니다

여기서 기대하는 것은 **새로운 발견이 아니라 상식의 재현**입니다.

- **자기 자신의 발현**이 상위에 오면 정상입니다. 유전자가 켜져 있어야 그걸 망가뜨렸을 때 타격이 있으니까요.
- **같은 계통의 다른 전사인자**가 잡히는 것도 정상입니다. 예를 들어 흑색종 마커끼리 함께 움직입니다.
- **패럴로그**(기능이 겹치는 형제 유전자, 예: `SMARCA2` ↔ `SMARCA4`)가 잡히면 특히 좋은 신호입니다.
  하나가 없으면 다른 하나에 의존하게 되는 관계라, 생물학적으로 알려진 패턴입니다.

이런 것이 하나도 안 나오고 무관해 보이는 유전자만 잡힌다면, 성능 수치가 좋아도 **파이프라인을 의심해야 합니다.**

## 6. 주의할 점 두 가지

### ① 사전선별은 반드시 학습 데이터 안에서만

19,215개를 전부 넣으면 너무 느려서, 상관이 높은 200개만 남기고 적합합니다.
이때 **전체 데이터로 200개를 고르면 시험 데이터의 정보가 새어 들어갑니다.**
성능이 좋아지는 방향으로 틀리기 때문에 눈으로는 절대 못 잡습니다.

아래는 그 함정을 실제로 재현한 것입니다. **정답이 완전한 난수**여서 원래는 아무것도 맞힐 수 없는데,
전체로 선별하면 상관이 생겨 버립니다.

In [8]:
rng = np.random.default_rng(0)
n, p = 100, 3000
Xr = rng.normal(size=(n, p))
yr = rng.normal(size=n)                      # X 와 아무 관계 없는 난수
tr, te = np.arange(70), np.arange(70, n)

honest, _ = enet_preselect_cv(Xr[tr], yr[tr], Xr[te], top_feat=30, seed=0)

leak = top_correlated(Xr, yr, 30)            # ← 전체 데이터로 선별 (하면 안 되는 것)
leaky, _ = enet_preselect_cv(Xr[tr][:, leak], yr[tr], Xr[te][:, leak], top_feat=30, seed=0)

r_h = pearson_cols(yr[te][:, None], honest[:, None], min_n=5)[0]
r_l = pearson_cols(yr[te][:, None], leaky[:, None], min_n=5)[0]
print(f"정답이 난수인 데이터에서:")
print(f"  올바른 방법 (학습 데이터로만 선별)  r = {r_h:+.3f}   ← 0 근처가 정상")
print(f"  누출 (전체 데이터로 선별)          r = {r_l:+.3f}   ← 없는 신호가 생김")

정답이 난수인 데이터에서:
  올바른 방법 (학습 데이터로만 선별)  r = -0.076   ← 0 근처가 정상
  누출 (전체 데이터로 선별)          r = +0.857   ← 없는 신호가 생김


`src/models/elastic_net.py` 의 `enet_preselect_cv` 는 학습 데이터만 받도록 만들어져 있고,
[`tests/test_elastic_net.py`](../tests/test_elastic_net.py) 가 이 성질을 자동으로 검사합니다.

### ② 수렴 경고

Elastic Net 은 반복 계산으로 답을 찾는데, 정해진 횟수 안에 충분히 수렴하지 못하면 경고가 뜹니다.
이 노트북 맨 위에서 경고를 껐습니다. 무시해도 되는 근거는 실측입니다 — `max_iter` 를 3,000에서
1,000으로, `tol` 을 1e-4에서 1e-3으로 **완화했더니 11.3배 빨라지면서 품질은 거의 그대로였습니다**
(중앙값 r 0.365 → 0.354, 유전자별 상관 0.986). 즉 원래 설정이 과했던 것이고,
경고가 뜨는 구간은 답에 거의 영향을 주지 않는 미세 조정이었습니다.

## 7. 정리

- **성능은 릿지가 낫습니다** (중앙값 r 0.404 vs 0.354). 속도는 23배 느립니다.
  단, 특징 수가 다른 설정이라 원리적 우열이 아니라 **이 설정에서의 결과**입니다.
- 그럼에도 쓰는 이유는 **근거를 읽을 수 있기 때문**입니다. 19,215개 중 수십 개만 남습니다.
- 남은 유전자가 상식(자기 자신·같은 계통·패럴로그)과 맞는지가 **파이프라인이 제대로 돌고 있다는 검증**입니다.

**실무 권장**: 성능 비교와 반복 실험은 빠른 릿지로, 최종 해석이 필요한 소수 유전자만 Elastic Net 으로.

### 더 볼 것

- 릿지 노트북: [`pilot_idea1_ridge.ipynb`](pilot_idea1_ridge.ipynb)
- 모델 코드: [`src/models/elastic_net.py`](../src/models/elastic_net.py)
- 누출 검증 테스트: [`tests/test_elastic_net.py`](../tests/test_elastic_net.py)